# Generation harness — RUN ON GOOGLE COLAB ONLY (T4 GPU)

**Do NOT run this on a laptop.** It needs an NVIDIA GPU and ~15 GB of GPU/system memory.

Works for a single run (Phase 0) or a whole grid (Phase 1+): it processes **every**
`generation_tasks*.jsonl` you upload and zips all completions for download.

Steps:
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload one or more `generation_tasks*.jsonl` files via the folder icon in the left sidebar
   (for Phase 1: all files from `results/phase1_tasks/`)
3. Runtime → Run all
4. `completions_bundle.zip` downloads automatically at the end — unzip into `results/`
   on your Mac so each file lands as `results/<config>/completions.jsonl` (then run scripts/place_completions.py)

In [ ]:
import torch, glob
assert torch.cuda.is_available(), (
    'No GPU detected! Go to Runtime -> Change runtime type -> select a GPU -> Save, then rerun.')
task_files = sorted(glob.glob('generation_tasks*.jsonl'))
assert task_files, (
    'No generation_tasks*.jsonl found! Upload task file(s) via the folder icon in the left sidebar.')
print('GPU:', torch.cuda.get_device_name(0), '| task files:', len(task_files))
for f in task_files:
    print('  -', f)

# Mount Drive so results survive even if the VM recycles after an unattended run.
# One interactive auth click needed HERE, at launch — do not skip for overnight runs.
DRIVE = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    print('Drive mounted — results will be saved to MyDrive automatically')
except Exception as e:
    print('Drive not mounted (continuing without):', e)

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = 'meta-llama/Llama-3.1-8B-Instruct'   # Llama arm (gated: needs HF_TOKEN secret)
ARM = 'llama'                                 # output suffix; '' for the primary Qwen arm
# For a Qwen (re)run use: MODEL = 'Qwen/Qwen2.5-7B-Instruct'; ARM = ''

try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get('HF_TOKEN'))
    print('HF login ok')
except Exception as e:
    print('no HF token found (fine for ungated models):', e)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto')
print('model loaded:', MODEL)

In [ ]:
import os

def run_file(task_file):
    name = task_file.replace('generation_tasks_', '').replace('generation_tasks', 'phase0_base').replace('.jsonl', '')
    suffix = f'__{ARM}' if ARM else ''
    out_file = f'completions_{name}{suffix}.jsonl'
    if os.path.exists(out_file):
        existing = sum(1 for _ in open(out_file))
    else:
        existing = 0
    tasks = [json.loads(l) for l in open(task_file)]
    if existing >= len(tasks):
        print(f'{out_file}: already complete, skipping')
        return out_file
    done = [json.loads(l) for l in open(out_file)] if existing else []
    for i, t in enumerate(tasks[existing:], start=existing):
        if t.get('pre_response'):
            text = t['pre_response']  # similarity gate: no LLM call
        else:
            msgs = [{'role': 'system', 'content': t['system']},
                    {'role': 'user', 'content': t['user']}]
            enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt',
                                          return_dict=True).to(model.device)
            out = model.generate(**enc, max_new_tokens=t.get('max_new_tokens', 150),
                                 do_sample=False, pad_token_id=tok.eos_token_id)
            text = tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        done.append({'qid': t['qid'], 'answerable': t['answerable'],
                     'gold_label': t['gold_label'], 'config': t.get('config', name),
                     'response_text': text, 'generator': MODEL})
        if (i + 1) % 25 == 0 or i + 1 == len(tasks):
            print(f'  {name}: {i+1}/{len(tasks)}')
            with open(out_file, 'w') as f:
                f.write('\n'.join(json.dumps(d) for d in done))
    return out_file

outputs = []
for tf in task_files:
    print('==', tf)
    outputs.append(run_file(tf))
print('all done:', len(outputs), 'files')

In [ ]:
import shutil, zipfile

with zipfile.ZipFile('completions_bundle.zip', 'w') as z:
    for o in outputs:
        z.write(o)

if DRIVE:
    shutil.copy('completions_bundle.zip', f'{DRIVE}/completions_bundle.zip')
    print(f'saved to Drive: {DRIVE}/completions_bundle.zip')

try:
    from google.colab import files
    files.download('completions_bundle.zip')
except Exception as e:
    print('browser download unavailable (fine if running unattended):', e)